# Part 4 — LLM-Powered Feature: Model Prediction Explanation Pipeline (Track C)

**Chosen track: (C) Model Prediction Explanation Pipeline.**

This loads the best-performing model from Part 3 (`best_model.pkl`, a tuned Random Forest pipeline predicting `smoker` status), runs three hand-crafted feature-vector inputs through it, and asks an LLM (via Groq's OpenAI-compatible chat completions API) to produce a structured, validated JSON explanation of each prediction for a non-technical stakeholder.

**Before running:** create a `.env` file in this folder with your API key:
```
GROQ_API_KEY=your_key_here
```
See `README.md` for step-by-step instructions on getting a free Groq API key.

In [1]:
import os
import re
import json
import joblib
import requests
import pandas as pd
from jsonschema import validate, ValidationError
from dotenv import load_dotenv

load_dotenv()  # loads GROQ_API_KEY (and optional GROQ_MODEL) from a local .env file

False

## Task: LLM API Connection

`call_llm` is a reusable function that POSTs to Groq's OpenAI-compatible `/chat/completions` endpoint. The API key is read from an environment variable (`GROQ_API_KEY`) -- it is never hardcoded.

In [2]:
LLM_API_KEY = os.environ.get("GROQ_API_KEY")
LLM_MODEL = os.environ.get("GROQ_MODEL", "llama-3.1-8b-instant")
LLM_URL = "https://api.groq.com/openai/v1/chat/completions"


def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    """Reusable function to call an OpenAI-compatible chat completions API (Groq)."""
    api_key = LLM_API_KEY
    if not api_key:
        print("ERROR: GROQ_API_KEY environment variable not set.")
        return None

    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    response = requests.post(LLM_URL, headers=headers, json=payload)

    if response.status_code != 200:
        print(f"LLM API error: status code {response.status_code}")
        print(response.text[:500])
        return None

    return response.json()["choices"][0]["message"]["content"]

### Test call_llm with a simple prompt

In [4]:
test_output = call_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt="Reply with only the word: hello",
    temperature=0.0,
    max_tokens=10,
)
print(f"Test output: {test_output!r}")

ERROR: GROQ_API_KEY environment variable not set.
Test output: None


## Task: PII Guardrail

Before every LLM call, a regex check blocks personally identifiable information (emails, phone numbers).

In [6]:
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))


def safe_call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    """Wraps call_llm with the mandatory PII guardrail."""
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None
    return call_llm(system_prompt, user_prompt, temperature=temperature, max_tokens=max_tokens)

### Guardrail demonstration: one input with PII (blocked), one clean (allowed through)

In [7]:
pii_input = "Please explain this record, contact me at jane.doe@example.com for questions."
clean_input = "Please explain this record for a general audience."

print(f"Testing PII input: {pii_input!r}")
result_pii = safe_call_llm("You are a helpful assistant.", pii_input)
print(f"Result: {result_pii}")

Testing PII input: 'Please explain this record, contact me at jane.doe@example.com for questions.'
Input blocked: PII detected.
Result: None


In [8]:
print(f"Testing clean input: {clean_input!r}")
result_clean = safe_call_llm("You are a helpful assistant.", clean_input, max_tokens=20)
print(f"Result: {result_clean}")

Testing clean input: 'Please explain this record for a general audience.'
ERROR: GROQ_API_KEY environment variable not set.
Result: None


## Task: JSON Schema for the Explanation Output

At least 5 required scalar fields, validated with `jsonschema.validate()`.

In [9]:
EXPLANATION_SCHEMA = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"},
    },
    "required": [
        "prediction_label", "confidence_level", "top_reason", "second_reason", "next_step",
    ],
}


def parse_and_validate(raw_response):
    """Strip, parse as JSON, validate against schema. Returns (result_dict, status_str)."""
    if raw_response is None:
        return None, "blocked_or_api_error"

    cleaned = raw_response.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json\n", "", 1).strip()

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e}")
        return {k: None for k in EXPLANATION_SCHEMA["required"]}, f"fail (JSONDecodeError: {e})"

    try:
        validate(instance=parsed, schema=EXPLANATION_SCHEMA)
    except ValidationError as e:
        print(f"ValidationError: {e.message}")
        return {k: None for k in EXPLANATION_SCHEMA["required"]}, f"fail (ValidationError: {e.message})"

    return parsed, "pass"

## Task: Load Model & Encode Records

`encode_record` assembles a raw feature dict into the exact column order the trained Part 3 pipeline expects. The pipeline itself (`SimpleImputer` -> `StandardScaler` -> `RandomForestClassifier`) handles imputation and scaling internally.

In [10]:
MODEL_COLUMNS = [
    "age", "bmi", "children", "exercise_freq", "prior_conditions",
    "sex_male", "region_northwest", "region_southeast", "region_southwest",
]

model = joblib.load("best_model.pkl")


def encode_record(features: dict) -> pd.DataFrame:
    row = pd.DataFrame([features])
    return row[MODEL_COLUMNS]

c:\Users\HP\-applied-ai-ml-capstone-\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\HP\-applied-ai-ml-capstone-\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\HP\-applied-ai-ml-capstone-\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from versio

## Prompt Design

**System prompt (zero-shot, per Track C):** instructs the LLM to produce a structured JSON explanation given feature values, predicted class, and predicted probability.

In [11]:
SYSTEM_PROMPT = (
    "You are a careful, factual model-explanation assistant for a health "
    "insurance analytics team. You will be given a data record's feature "
    "values, a machine learning model's predicted class, and the model's "
    "predicted probability for that class. Your job is to produce a short, "
    "plain-language explanation of the prediction for a non-technical "
    "stakeholder. Output ONLY a single valid JSON object -- no markdown code "
    "fences, no extra commentary before or after it. The JSON object must "
    "have exactly these fields: "
    '"prediction_label" (string, e.g. "likely smoker" or "likely non-smoker"), '
    '"confidence_level" (string, one of "low", "medium", "high", based on how '
    "far the predicted probability is from 0.5), "
    '"top_reason" (string, the single feature most likely driving this '
    "prediction, in plain language), "
    '"second_reason" (string, the second most likely contributing feature), '
    '"next_step" (string, a brief, sensible recommended next action for the '
    "stakeholder, e.g. offering a wellness check-in). "
    "Base your reasoning only on the feature values provided; do not invent "
    "facts not present in the input."
)


def build_user_prompt(features: dict, predicted_class: int, probability: float) -> str:
    class_label = "smoker" if predicted_class == 1 else "non-smoker"
    return (
        "Feature values:\n"
        f"{json.dumps(features, indent=2)}\n\n"
        f"Predicted class: {predicted_class} ({class_label})\n"
        f"Predicted probability of class 1 (smoker): {probability:.4f}\n\n"
        "Produce the JSON explanation now."
    )

## Task: Demonstrate the Feature End-to-End (Three Hand-Crafted Inputs)

In [12]:
test_records = [
    {  # Record 1: young, low BMI, no kids, high exercise, no prior conditions
        "age": 24, "bmi": 21.5, "children": 0, "exercise_freq": 3, "prior_conditions": 0,
        "sex_male": True, "region_northwest": False, "region_southeast": False, "region_southwest": False,
    },
    {  # Record 2: older, high BMI, several kids, no exercise, multiple prior conditions
        "age": 57, "bmi": 35.2, "children": 3, "exercise_freq": 0, "prior_conditions": 3,
        "sex_male": False, "region_northwest": False, "region_southeast": True, "region_southwest": False,
    },
    {  # Record 3: middle-aged, moderate BMI, one child, moderate exercise
        "age": 40, "bmi": 27.8, "children": 1, "exercise_freq": 2, "prior_conditions": 1,
        "sex_male": True, "region_northwest": True, "region_southeast": False, "region_southwest": False,
    },
]

demonstration_rows = []
for i, features in enumerate(test_records):
    encoded = encode_record(features)
    pred_class = int(model.predict(encoded)[0])
    pred_proba = float(model.predict_proba(encoded)[0, 1])

    user_prompt = build_user_prompt(features, pred_class, pred_proba)
    print(f"--- Record {i+1} ---")
    print(f"Features: {features}")
    print(f"Predicted class: {pred_class} | Predicted probability: {pred_proba:.4f}")

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        raw_response = None
        validation_status = "blocked"
        parsed = {k: None for k in EXPLANATION_SCHEMA["required"]}
    else:
        raw_response = call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.0, max_tokens=300)
        print(f"Raw LLM response: {raw_response}")
        parsed, validation_status = parse_and_validate(raw_response)

    print(f"Validation status: {validation_status}")
    print(f"Parsed result: {parsed}\n")

    demonstration_rows.append({
        "record_index": i + 1, "features": features, "predicted_class": pred_class,
        "predicted_probability": pred_proba, "raw_response": raw_response,
        "parsed": parsed, "validation_status": validation_status,
    })

--- Record 1 ---
Features: {'age': 24, 'bmi': 21.5, 'children': 0, 'exercise_freq': 3, 'prior_conditions': 0, 'sex_male': True, 'region_northwest': False, 'region_southeast': False, 'region_southwest': False}
Predicted class: 0 | Predicted probability: 0.2056
ERROR: GROQ_API_KEY environment variable not set.
Raw LLM response: None
Validation status: blocked_or_api_error
Parsed result: None

--- Record 2 ---
Features: {'age': 57, 'bmi': 35.2, 'children': 3, 'exercise_freq': 0, 'prior_conditions': 3, 'sex_male': False, 'region_northwest': False, 'region_southeast': True, 'region_southwest': False}
Predicted class: 0 | Predicted probability: 0.2738
ERROR: GROQ_API_KEY environment variable not set.
Raw LLM response: None
Validation status: blocked_or_api_error
Parsed result: None

--- Record 3 ---
Features: {'age': 40, 'bmi': 27.8, 'children': 1, 'exercise_freq': 2, 'prior_conditions': 1, 'sex_male': True, 'region_northwest': True, 'region_southeast': False, 'region_southwest': False}
Pred

## Task: Temperature A/B Comparison (temp=0 vs temp=0.7)

In [13]:
temp_comparison_rows = []
for i, features in enumerate(test_records):
    encoded = encode_record(features)
    pred_class = int(model.predict(encoded)[0])
    pred_proba = float(model.predict_proba(encoded)[0, 1])
    user_prompt = build_user_prompt(features, pred_class, pred_proba)

    out_temp0 = call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.0, max_tokens=300)
    out_temp07 = call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.7, max_tokens=300)

    print(f"--- Record {i+1} ---")
    print(f"temp=0.0 output: {out_temp0}")
    print(f"temp=0.7 output: {out_temp07}\n")

    temp_comparison_rows.append({
        "record_index": i + 1, "temp0_output": out_temp0, "temp07_output": out_temp07,
    })

ERROR: GROQ_API_KEY environment variable not set.
ERROR: GROQ_API_KEY environment variable not set.
--- Record 1 ---
temp=0.0 output: None
temp=0.7 output: None

ERROR: GROQ_API_KEY environment variable not set.
ERROR: GROQ_API_KEY environment variable not set.
--- Record 2 ---
temp=0.0 output: None
temp=0.7 output: None

ERROR: GROQ_API_KEY environment variable not set.
ERROR: GROQ_API_KEY environment variable not set.
--- Record 3 ---
temp=0.0 output: None
temp=0.7 output: None



## Auto-Generate README Tables

Run this cell last. It prints ready-to-paste Markdown tables built from this run's **real** results -- copy the output below directly into `README.md` in place of the placeholder tables.

In [16]:
print("### Three-Row Demonstration Table\n")
print("| Input (Record) | LLM Output | Valid JSON | Pass/Block |")
print("|---|---|---|---|")
for row in demonstration_rows:
    feat_short = f"age={row['features']['age']}, bmi={row['features']['bmi']}, smoker_pred={row['predicted_class']} (p={row['predicted_probability']:.3f})"
    llm_out_short = (row["raw_response"] or "BLOCKED")[:120].replace("|", "\\|").replace("\n", " ")
    valid_flag = "pass" if row["validation_status"] == "pass" else "fail"
    pass_block = "blocked" if row["validation_status"] == "blocked" else "pass"
    print(f"| {feat_short} | {llm_out_short} | {valid_flag} | {pass_block} |")

print("\n### Temperature A/B Comparison Table\n")
print("| Input (Record) | Output at temp=0 | Output at temp=0.7 | Key Difference |")
print("|---|---|---|---|")
for row in temp_comparison_rows:
    t0 = (row["temp0_output"] or "None")[:100].replace("|", "\\|").replace("\n", " ")
    t07 = (row["temp07_output"] or "None")[:100].replace("|", "\\|").replace("\n", " ")
    same = "Identical" if row["temp0_output"] == row["temp07_output"] else "Differs"
    print(f"| Record {row['record_index']} | {t0} | {t07} | {same} |")

print("\n### Guardrail Test Results\n")
print("| Input | PII Detected | LLM Called |")
print("|---|---|---|")
print(f'| "{pii_input}" | Yes | No (blocked) |')
print(f'| "{clean_input}" | No | Yes |')

### Three-Row Demonstration Table

| Input (Record) | LLM Output | Valid JSON | Pass/Block |
|---|---|---|---|
| age=24, bmi=21.5, smoker_pred=0 (p=0.206) | BLOCKED | fail | pass |
| age=57, bmi=35.2, smoker_pred=0 (p=0.274) | BLOCKED | fail | pass |
| age=40, bmi=27.8, smoker_pred=0 (p=0.172) | BLOCKED | fail | pass |

### Temperature A/B Comparison Table

| Input (Record) | Output at temp=0 | Output at temp=0.7 | Key Difference |
|---|---|---|---|
| Record 1 | None | None | Identical |
| Record 2 | None | None | Identical |
| Record 3 | None | None | Identical |

### Guardrail Test Results

| Input | PII Detected | LLM Called |
|---|---|---|
| "Please explain this record, contact me at jane.doe@example.com for questions." | Yes | No (blocked) |
| "Please explain this record for a general audience." | No | Yes |
